## Notebook and DF prep

In [1]:
import pandas as pd
from dotenv import dotenv_values
from sqlalchemy import create_engine
import numpy as np
import hashlib
from sqlalchemy import text

config = dotenv_values('../.env')
engine = create_engine(
    f"postgresql+psycopg2://{config['DB_USER']}:{config['DB_PASSWORD']}"
    f"@{config['DB_HOST']}:{config['DB_PORT']}/{config['DB_NAME']}"
)

In [2]:
# the text columns where undertone words can appear, plus the columns for the shade key
shades = pd.read_sql(
    """
    select brand_name, product_name, product_url, hex_original,
           shade_name, shade_number, shade_description
    from dev.stg_shades
    """,
    engine,
)
shades.shape

(6791, 7)

## Words in shade descriptions

We start with shade_description, where most undertone words should be. Here’s what the counting does, bottom up, with two made-up descriptions:

shade_description
Fair with cool pink undertones
Medium, warm golden

In [3]:
# .dropna() skips shades without a description 
# .str.lower() makes everything lowercase, so “Warm” and “warm” count as one word: fair with cool pink undertones, medium, warm golden.
# .str.findall(r'[a-z]+') cuts each text into words. [a-z]+ means “one or more letters in a row”, so commas, numbers and spaces split the words: [fair, with, cool, pink, undertones] and [medium, warm, golden].
# .explode() turns each list into separate rows, one word per row, for 8 rows here.
# .value_counts() counts how often each word appears, from most to least.

desc_words = (
    shades['shade_description']
    .dropna()
    .str.lower()
    .str.findall(r'[a-z]+')
    .explode()
    .value_counts()
)
desc_words.head(60)

shade_description
undertones    3412
skin          2530
with          2518
neutral       2390
medium        2372
w             2109
light         1800
warm          1590
deep          1455
undertone     1436
for           1384
golden        1148
tan           1052
cool          1013
fair           991
beige          786
to             661
dark           618
yellow         566
n              552
very           549
pink           490
peach          409
a              336
olive          324
rich           313
selected       311
red            302
c              273
ivory          243
brown          227
sand           213
of             205
honey          202
rosy           201
tones          193
y              175
porcelain      150
natural        150
subtle         146
out            131
stock          131
f              128
nude           115
g              105
p               95
almond          86
shade           84
nc              84
deepest         83
tone            83
peachy       

In [4]:
# in order to get a better understanding what the letters may represent
# this code shows five example descriptions for each letter
# rf'\b{letter}\b' is a pattern for “this letter as a word on its own”
# \b means a word boundary, so it finds the “w” in “w/” or in “2W”, but not the w in “warm”
# case=False finds “W” and “w” alike 
# na=False treats empty descriptions as “no match” instead of giving an error
# .head(5) shows five examples per letter
# to_string(index=False) prints them without row numbers, so they’re easier to read
for letter in ['w', 'n', 'c', 'y', 'f', 'g', 'p', 'r', 'b']:
    examples = shades.loc[
        shades['shade_description'].str.contains(rf'\b{letter}\b', case=False, na=False),
        ['brand_name', 'shade_description'],
    ].head(5)
    print(f'--- {letter} ---')
    print(examples.to_string(index=False))

--- w ---
     brand_name                                                     shade_description
BECCA Cosmetics      Shell 1W1 (very fair beige w/ golden, warm undertones)  selected
BECCA Cosmetics        Linen 1N1 (very fair beige w/ neutral undertones) OUT OF STOCK
BECCA Cosmetics              Alabaster 1C1 (very fair beige w/ rosy, cool undertones)
BECCA Cosmetics Porcelain 1N2 (fair beige w/ peachy, neutral undertones) OUT OF STOCK
BECCA Cosmetics                       Ivory 1C2 (fair beige w/ rosy, cool undertones)
--- n ---
      brand_name                                            shade_description
KVD Vegan Beauty Medium 54 N (medium sesame with neutral undertone)  selected
KVD Vegan Beauty          Light 40 N (near-white for mixing & artistry looks)
KVD Vegan Beauty               Light 41 N (fair ivory with neutral undertone)
KVD Vegan Beauty           Light 42 N (fair porcelain with neutral undertone)
KVD Vegan Beauty              Light 45 N (light beige with neutral underto

In [5]:
#(?:w/|with): “w/” or “with”. The (?: ) groups the two options without capturing them.
# \s+: one or more spaces.
# ([a-z ,\-]+?): the part we want, captured by the brackets. It can contain letters, spaces, commas and hyphens. The ? at the end means “as short as possible”, so it stops at the first “undertone”
# \s+undertones?: spaces, then “undertone”, with an optional “s” (the ? after the s)
# .str.extract(pattern)[0] puts the captured part into a new column
# Rows without the pattern get an empty value


pattern = r'(?:w/|with)\s+([a-z ,\-]+?)\s+undertones?'

shades['undertone_phrase'] = (
    shades['shade_description']
    .str.lower()
    .str.extract(pattern)[0]
)

In [6]:
# how many shades does the pattern cover?
print(shades['shade_description'].notna().sum(), 'shades have a description')
print(shades['undertone_phrase'].notna().sum(), 'got an undertone phrase')

# which phrases come up, most common first
shades['undertone_phrase'].value_counts().head(40)

6791 shades have a description
4069 got an undertone phrase


undertone_phrase
neutral              960
golden               386
warm                 315
yellow               259
cool                 241
pink                 239
peach                159
red                  110
rosy                  95
warm yellow           56
warm golden           55
a neutral             46
olive                 46
warm olive            33
warm peach            32
cool pink             30
neutral golden        26
neutral olive         23
a golden              22
peachy                21
subtle peach          20
slightly red          20
subtle red            19
subtle yellow         18
a warm                17
cool red              17
a pink                16
slightly golden       15
golden or olive       14
a peach               14
cool neutral          14
warm neutral          14
a cool                13
cool peach            13
warm, peachy          13
a yellow              13
neutral warm          13
neutral to warm       13
slightly peach        12
neutral 

In [7]:
pd.set_option('display.max_colwidth', None)

In [8]:
no_phrase = shades[shades['undertone_phrase'].isna()]

# which brands describe shades without the "w/ … undertones" pattern?
print(no_phrase['brand_name'].value_counts().head(15))

# 20 random examples to see what their descriptions look like
no_phrase[['brand_name', 'shade_description']].sample(20, random_state=1)

brand_name
Clinique              225
L'Oréal               153
Dior                  134
ULTA                  132
Maybelline            129
HOURGLASS             109
Lancôme               107
Milani                 91
CoverGirl              84
SEPHORA COLLECTION     83
TOM FORD               79
Yves Saint Laurent     69
Laura Mercier          58
Estée Lauder           57
Armani Beauty          57
Name: count, dtype: int64


,brand_name,shade_description
3229,bareMinerals,Neutral Ivory 06 (for very light neutral skin with a peach hue)
2543,Milani,Light Beige
3004,Maybelline,Soft Tan
2810,L.A. Girl,Espresso
6712,Natasha Denona,50 MW medium warm
6511,Shiseido,"530 Henna balanced tone for deep skin, neutral undertone"
2519,Almay,Medium
3300,Physicians Formula,Fair to Light
2430,It Cosmetics,Fair (porcelain) OUT OF STOCK
5800,Yves Saint Laurent,BR40 Light/medium neutral to pink undertone


In [9]:

no_phrase = shades[shades['undertone_phrase'].isna()].copy()

# undertone words anywhere in the description
undertone_words = ['neutral', 'warm', 'cool', 'golden', 'yellow', 'pink',
                   'peach', 'peachy', 'olive', 'red', 'rosy']
word_pattern = r'\b(' + '|'.join(undertone_words) + r')\b'

no_phrase['any_word'] = (
    no_phrase['shade_description']
    .str.lower()
    .str.findall(word_pattern)
    .str.join(' ')
)

print((no_phrase['any_word'] != '').sum(), 'of', len(no_phrase), 'contain an undertone word somewhere')

1605 of 2722 contain an undertone word somewhere


In [10]:
for brand in ['Clinique', "L'Oréal", 'Dior']:
    print(f'--- {brand} ---')
    print(no_phrase.loc[no_phrase['brand_name'] == brand, 'shade_description'].head(5).to_string(index=False))

--- Clinique ---
03 Light  selected
           01 Fair
   02 Neutral Fair
        04 Neutral
         05 Medium
--- L'Oréal ---
Pearl  selected
      Porcelain
        Vanilla
     True Beige
   Golden Beige
--- Dior ---
                                   1.5 Neutral
 1 Neutral (100) Fair skin, neutral undertones
       1 Cool (104) Fair skin, cool undertones
                                        1 Warm
1 Warm (101) Fair skin, warm yellow undertones


In [11]:
nothing = no_phrase[no_phrase['any_word'] == ''].copy()

In [12]:
print(nothing['brand_name'].value_counts().head(10))

brand_name
L'Oréal               138
Maybelline            113
Lancôme                63
Milani                 61
CoverGirl              59
KVD Vegan Beauty       51
Revlon                 46
SEPHORA COLLECTION     45
Wet n Wild             43
Clinique               36
Name: count, dtype: int64


In [13]:
nothing[['brand_name', 'shade_name', 'shade_number', 'shade_description']].sample(15, random_state=1)

,brand_name,shade_name,shade_number,shade_description
2428,Estée Lauder,Honey Bronze,4W1,4W1 Honey Bronze
6535,rms beauty,NaN,88,88 a rich auburn
2113,Maybelline,True Beige,222,True Beige 222
474,Perricone MD,Ivory,NaN,Ivory
2139,Revlon,Caramel,400,400 Caramel
1144,J.Cat Beauty,Caramel,NaN,Caramel
3178,CoverGirl,Tan/Rich,590,Tan/Rich 590
738,J.Cat Beauty,Soft Beige,NaN,Soft Beige
1066,It Cosmetics,Light,NaN,Light (fair light)
1997,Maybelline,Java,NaN,Java


In [14]:
# has_code = … checks, for each of the 1,117 rows, whether shade_number contains at least one letter
# [A-Za-z] means any letter, small or capital, so “222” gives False and “4W1” gives True. 
# na=False treats empty shade numbers as False.
# print(has_code.sum(), …) counts the Trues, since True counts as 1.
# print(nothing.loc[has_code, 'brand_name'].value_counts()) shows which brands those codes belong to
# Mostly Estée Lauder, with its known system, or many brands with unknown ones?

has_code = nothing['shade_number'].str.contains(r'[A-Za-z]', na=False)
print(has_code.sum(), 'shades have a letter code in shade_number')
print(nothing.loc[has_code, 'brand_name'].value_counts())

114 shades have a letter code in shade_number
brand_name
CoverGirl             26
Dermablend            17
Guerlain              15
Estée Lauder          14
FLOWER Beauty         10
Laura Mercier          8
MAKE UP FOR EVER       6
Tarte                  5
Physicians Formula     4
MAC                    3
jane iredale           2
florence by mills      1
Lancôme                1
Yves Saint Laurent     1
Erborian               1
Name: count, dtype: int64


In [15]:
# let's see how those one letter shade descriptions look like

# coded = … keeps only the 114 rows with a letter code, and three columns: brand, code and shade name
# coded.groupby('brand_name').head(3) takes the first 3 rows per brand, so every brand shows up, not just the biggest
# .sort_values('brand_name') puts the brands in alphabetical order, so each brand’s rows sit together.
coded = nothing.loc[has_code, ['brand_name', 'shade_number', 'shade_name']]
coded.groupby('brand_name').head(3).sort_values('brand_name')

,brand_name,shade_number,shade_name
517,CoverGirl,M50,Soft Tan
649,CoverGirl,M60,Natural Beige
673,CoverGirl,L10,Fair Porcelain
1440,Dermablend,35C,Caramel
1441,Dermablend,10C,Nude
1442,Dermablend,20W,Cream
6122,Erborian,SPF20,Erborian BB Cream Tinted Moisturizer Broad Spectrum
2416,Estée Lauder,2N1,Desert Beige
2414,Estée Lauder,3C4,Medium/Deep
2415,Estée Lauder,1N1,Ivory Nude


In [16]:
# (?<![A-Za-z]) means “not right after another letter”. 
# This keeps MAC’s “NC15” out, because its C comes right after N, and MAC gets its own rule.
# [CNW] is one of the three letters.
# (?![A-Za-z]) means “not right before another letter”, so the letter stands among numbers, as in “2N1” or “35C”.
# |Dp[CNW] means “or Dp followed by C, N or W”, for Physicians Formula, where the letter comes right after “Dp”.

cnw = coded['shade_number'].str.contains(r'(?<![A-Za-z])[CNW](?![A-Za-z])|Dp[CNW]')
print(cnw.sum(), 'of', len(coded), 'coded shades follow the C/N/W pattern')
print(coded.loc[cnw, 'brand_name'].value_counts())

59 of 114 coded shades follow the C/N/W pattern
brand_name
Dermablend            17
Guerlain              15
Estée Lauder          14
Laura Mercier          8
Physicians Formula     4
Lancôme                1
Name: count, dtype: int64


In [17]:
# MAC has one of the most confusing system for shade description, they require special attention
mac = shades[shades['brand_name'] == 'MAC']
print(len(mac), 'MAC shades')

185 MAC shades


In [18]:
# This keeps only MAC’s rows and counts them
print(mac['shade_number'].str.extract(r'^([A-Z]+)')[0].value_counts(dropna=False))

0
NC    84
NW    70
C     21
N     10
Name: count, dtype: int64


In [19]:
# let's look at 20 real MAC examples
mac[['product_name', 'shade_number', 'shade_description', 'undertone_phrase']].head(20)

,product_name,shade_number,shade_description,undertone_phrase
1159,Studio Fix Soft Matte Foundation Stick,NC20,NC20 (light beige w/ neutral golden undertone for light skin) selected,neutral golden
1160,Studio Fix Soft Matte Foundation Stick,NC13,NC13 (fair beige w/ golden undertone for fair skin),golden
1161,Studio Fix Soft Matte Foundation Stick,NW13,NW13 (fair beige w/ neutral undertone for light skin),neutral
1162,Studio Fix Soft Matte Foundation Stick,NC15,NC15 (light beige w/ golden undertone for light skin),golden
1163,Studio Fix Soft Matte Foundation Stick,NW15,NW15 (light beige w/ neutral undertone for light skin),neutral
1164,Studio Fix Soft Matte Foundation Stick,NC16,NC16 (light beige w/ peachy undertone for light skin),peachy
1165,Studio Fix Soft Matte Foundation Stick,NC18,NC18 (beige w/ neutral undertone w/ light skin),neutral
1166,Studio Fix Soft Matte Foundation Stick,NW18,NW18 (beige w/ neutral undertone for light skin),neutral
1167,Studio Fix Soft Matte Foundation Stick,NW20,NW20 (warm neutral beige w/ rosy undertone for light to medium skin),rosy
1168,Studio Fix Soft Matte Foundation Stick,NW22,NW22 (warm beige w/ rosy undertone for light to medium skin),rosy


In [20]:
# str.match(r'^[CN]\d', …) keeps shade numbers that start (^) with C or N followed directly by a digit (\d), like “C3” or “N5”. 
# “NC15” doesn’t match, because after the N comes a C, not a digit.
# na=False treats empty shade numbers as no match.
mac_c_n = mac[mac['shade_number'].str.match(r'^[CN]\d', na=False)]
mac_c_n[['product_name', 'shade_number', 'shade_description', 'undertone_phrase']]

,product_name,shade_number,shade_description,undertone_phrase
1540,Studio Fix Powder Plus Foundation,C2,C2 (very fair w/ golden undertone for light skin),golden
1543,Studio Fix Powder Plus Foundation,N3,N3 (very fair w/ rosy undertone for light skin),rosy
1544,Studio Fix Powder Plus Foundation,N4,N4 (fair w/ neutral undertone for light skin),neutral
1550,Studio Fix Powder Plus Foundation,C3,C3 (beige w/ neutral golden undertone for light to medium skin),neutral golden
1551,Studio Fix Powder Plus Foundation,C3.5,C3.5 (light peach w/ golden undertone for light to medium skin),golden
1552,Studio Fix Powder Plus Foundation,C4,C4 (peachy golden w/ golden undertone for light to medium skin),golden
1553,Studio Fix Powder Plus Foundation,C30,C30 (light golden olive w/ golden undertone for light to medium skin),golden
1556,Studio Fix Powder Plus Foundation,C4.5,C4.5 (peachy golden w/ neutral undertone for medium skin),neutral
1557,Studio Fix Powder Plus Foundation,C5,C5 (peachy beige w/ golden undertone for medium skin),golden
1558,Studio Fix Powder Plus Foundation,C5.5,C5.5 (mid tone peach w/ peachy undertone for medium skin),peachy


from the MAC result I see that some of the undertones weren't picked up (Studio Fix Fluid SPF 15 Foundation	C4 and C3.5)
I need to build a better logic to capture undertones as different brands have all different solution

In [21]:
# This loads Python’s pattern module, needed for re.findall below.
import re


In [22]:
# these are the words allowed into the brand term: the undertone words from earlier, 
# plus the connectors and intensity words
# set() makes looking a word up quick.
keep = set(undertone_words + [
    'gold', 'beige', 'brown', 'rose', 'rosey', 'bronze', 'orange', 'plum', 'reddish',
    'amber', 'mocha', 'sienna', 'mauve', 'auburn', 'ochre', 'caramel',
    'lightpink', 'goldenolive', 'blue', 'green',
    'to', 'or', 'and', 'subtle', 'slightly',
])

In [23]:
def undertone_term(text):
    if pd.isna(text):
        return None
    match = re.search(r'^(.*?)\bundertones?\b', text.lower())
    if not match:
        return None
    words = re.findall(r'[a-z]+', match.group(1))
    term = []
    for word in reversed(words):
        if word in keep:
            term.insert(0, word)
        else:
            break
    while term and term[0] in ('to', 'or', 'and'):
        term.pop(0)
    while term and term[-1] in ('to', 'or', 'and'):
        term.pop()
    term = [w for i, w in enumerate(term) if i == 0 or w != term[i - 1]]
    return ' '.join(term) or None

In [24]:
shades['undertone_term'] = shades['shade_description'].apply(undertone_term)
print(shades['undertone_term'].notna().sum(), 'shades with an undertone term (first version: 4,069; second: 4,552)')

4836 shades with an undertone term (first version: 4,069; second: 4,552)


In [25]:
# shades['undertone_phrase'].isna() means the first version found nothing.
# & means “and”.
# shades['undertone_term'].notna() means the new version found something.
# Together, they’re the 483 newly found shades, 
# and sample(15, random_state=1) shows 15 of them, the same 15 each rerun
new_ones = shades[shades['undertone_phrase'].isna() & shades['undertone_term'].notna()]
new_ones[['brand_name', 'shade_description', 'undertone_term']].sample(15, random_state=1)

,brand_name,shade_description,undertone_term
2776,Lancôme,435 Bisque W (medium to deep skin tones w/ warm/yellow undertones),warm yellow
5203,Yves Saint Laurent,B30 Almond light. neutral undertone,neutral
5845,Lancôme,025W Soft Beige for light to medium skin with warm/yellow undertones,warm yellow
6499,Shiseido,"160 Shell balanced tone for fair-light skin, neutral undertone",neutral
3492,HOURGLASS,"Ivory (light medium, warm undertone)",warm
1877,Clinique,"70 Vanilla (moderately fair, cool-neutral undertones)",cool neutral
2460,Clinique,"WN 30 Biscuit (very fair, warm-neutral undertones)",warm neutral
5211,Yves Saint Laurent,B45 Bisque medium. neutral undertone,neutral
5437,TOM FORD,"7.2 Sepia medium-dark, warm golden undertone",warm golden
1956,Clinique,"WN 124 Sienna (deep, warm-neutral undertones)",warm neutral


In [26]:
# both = … keeps the shades the first version already found
# changed = … keeps those where the old and the new text aren’t the same (!= means “not equal”)
# The print counts them
# value_counts().head(20) shows the 20 most common old → new pairs, like “a neutral” → “neutral” or “golden, warm” → “golden warm”
# Those changes are expected. Anything that lost an undertone word would be a problem


both = shades[shades['undertone_phrase'].notna()]
changed = both[both['undertone_phrase'] != both['undertone_term']]
print(len(changed), 'shades where old and new differ')
changed[['undertone_phrase', 'undertone_term']].value_counts().head(20)

465 shades where old and new differ


undertone_phrase        undertone_term       
a neutral               neutral                  46
a golden                golden                   22
a warm                  warm                     17
a pink                  pink                     16
a peach                 peach                    14
a cool                  cool                     13
warm, peachy            warm peachy              13
a yellow                yellow                   13
neutral, subtle golden  neutral subtle golden     9
very warm               warm                      9
an olive                olive                     8
golden, warm            golden warm               7
a mix of cool and warm  cool and warm             7
very pink               pink                      7
beigey-pink             pink                      7
a cool golden           cool golden               6
rosy, cool              rosy cool                 6
cool, pink              cool pink                 6
golden-peach      

In [27]:
lost = shades[shades['undertone_phrase'].notna() & shades['undertone_term'].isna()]
print(len(lost), 'shades the first version found but the new one misses')
lost[['brand_name', 'shade_description', 'undertone_phrase']].head(20)

10 shades the first version found but the new one misses


,brand_name,shade_description,undertone_phrase
669,Urban Decay Cosmetics,80WR (deep warm with a rich undertone),a rich
2187,Juice Beauty,10 Naked Beige (for light to medium skin with balanced undertones),balanced
2578,NYX Professional Makeup,Soft Beige (medium w/ light undertone) selected,light
3050,Urban Decay Cosmetics,60WR (medium-dark warm w/ rich undertone),rich
3056,Urban Decay Cosmetics,70WR (dark warm w/ rich undertone),rich
3065,Urban Decay Cosmetics,80WR (deep warm w/ rich undertone),rich
3068,Urban Decay Cosmetics,90WR (ultra-deep warm w/ rich undertone),rich
3070,Urban Decay Cosmetics,91WR (ultra-deep warm w/ rich undertone),rich
3784,Exa,310 Brandy (medium with golden tan undertones),golden tan
6426,Armani Beauty,1 Nude Glow for fair skin with all undertones,all


It seems that some brands are very creative with their shade descriptions and go beyond the usual words (amber, bronze etc)

In [28]:
# word_before = …: for each description, lowercase it, 
# then str.extract captures one word (([a-z]+)) that is followed by optional spaces, commas or hyphens ([\s,\-]*) 
# and then “undertone(s)”. For “warm beige undertone”, that’s “beige”
# ~word_before.isin(keep): isin(keep) checks whether the word is already on the keep list, 
# and the ~ turns that around, so it keeps only words that are not on it yet
# .value_counts().head(30): counts those words, most common first, and shows the top 30


word_before = (
    shades['shade_description']
    .str.lower()
    .str.extract(r'([a-z]+)[\s,\-]*undertones?')[0]
)
word_before[~word_before.isin(keep)].value_counts().head(30)

0
rich        6
skin        1
balanced    1
light       1
tan         1
all         1
Name: count, dtype: int64

In [29]:
shades.loc[word_before.isin(['blue', 'green']), ['brand_name', 'product_name', 'shade_description']]

,brand_name,product_name,shade_description
2820,Morphe,Fluidity Full-Coverage Foundation,F1.90 Olive (light w/ neutral green undertones))
2828,Morphe,Fluidity Full-Coverage Foundation,F2.50 Olive (light medium w/ yellow-green undertones)
2837,Morphe,Fluidity Full-Coverage Foundation,F3.20 Olive (medium w/ golden-green undertones)
2845,Morphe,Fluidity Full-Coverage Foundation,F3.100 Olive (tan w/ rich golden-green undertones)
2858,Morphe,Fluidity Full-Coverage Foundation,F4.110 Olive (rich w/ neutral green undertones)
2869,Morphe,Fluidity Full-Coverage Foundation,F5.90 Olive (deepest w/ rich green undertones)
4401,Rare Beauty by Selena Gomez,Liquid Touch Weightless Foundation,570 N deep with neutral blue undertones
5045,Bobbi Brown,Skin Long-Wear Weightless Foundation SPF 15,Cool Walnut (C-096) deep brown with red-blue undertones for deep skin
5049,Bobbi Brown,Skin Long-Wear Weightless Foundation SPF 15,Cool Chestnut (C-106) rich brown with red-blue undertones for rich skin
5052,Bobbi Brown,Skin Long-Wear Weightless Foundation SPF 15,Cool Espresso (C-116) rich brown with red-blue undertones for deepest skin


In [30]:
undertone_only = set(undertone_words + ['gold'])
any_word_pattern = r'\b(?:' + '|'.join(undertone_only) + r')\b'

In [31]:
has_phrase = shades['undertone_term'].notna()
has_word = shades['shade_description'].str.lower().str.contains(any_word_pattern, na=False)
has_code = (
    shades['shade_number'].str.contains(r'(?<![A-Za-z])[CNW](?![A-Za-z])|Dp[CNW]', na=False)
    & (shades['brand_name'] != 'MAC')
)

In [32]:
shades['undertone_source'] = np.select(
    [has_phrase, has_word, has_code],
    ['phrase', 'word', 'code'],
    default='none',
)

In [33]:
shades['undertone_source'].value_counts()

undertone_source
phrase    4836
none      1049
word       847
code        59
Name: count, dtype: int64

In [34]:
id_cols = ['product_url', 'hex_original', 'shade_name', 'shade_number']
check = pd.read_sql(f"select {', '.join(id_cols)} from dev.stg_shades", engine)
print(check.duplicated(subset=id_cols, keep=False).sum(), 'shades share their combination with another shade')

24 shades share their combination with another shade


In [35]:
look_cols = id_cols + ['brand_name', 'shade_description', 'l_star']
dups = pd.read_sql(f"select {', '.join(look_cols)} from dev.stg_shades", engine)
dups = dups[dups.duplicated(subset=id_cols, keep=False)].sort_values(id_cols)
dups[['brand_name', 'shade_name', 'shade_number', 'hex_original', 'shade_description', 'l_star']]

,brand_name,shade_name,shade_number,hex_original,shade_description,l_star
5167,Tarte,Light,NaN,#DDAF93,Light,74.913898
5175,Tarte,Light,NaN,#DDAF93,Light,74.913898
5166,Tarte,Fair,NaN,#F6E5D1,Fair,91.790797
5174,Tarte,Fair,NaN,#F6E5D1,Fair,91.790797
4114,Armani Beauty,NaN,5,#C9A07F,"5 light, neutral",68.818946
4148,Armani Beauty,NaN,5,#C9A07F,"5 light, neutral",68.818946
4082,Armani Beauty,NaN,5.5,#CD9E83,"5.5 medium, golden",68.804699
4150,Armani Beauty,NaN,5.5,#CD9E83,"5.5 medium, peach",68.804699
4098,Armani Beauty,NaN,5.75,#CEA180,"5.75 light to medium, golden",69.592111
4151,Armani Beauty,NaN,5.75,#CEA180,"5.75 light to medium, golden",69.592111


In [36]:
def make_shade_key(row):
    parts = [row['product_url'], row['hex_original'], row['shade_name'], row['shade_number']]
    text = '|'.join('' if pd.isna(p) else p for p in parts)
    return hashlib.md5(text.encode('utf-8')).hexdigest()

In [37]:
shades['shade_key'] = shades.apply(make_shade_key, axis=1)
print(shades['shade_key'].nunique(), 'different keys for', len(shades), 'shades')

6779 different keys for 6791 shades


In [38]:
desc = shades['shade_description'].str.lower()
word_found = desc.str.findall(any_word_pattern).apply(lambda ws: ' '.join(dict.fromkeys(ws)) or None)

code_cnw = shades['shade_number'].str.extract(r'(?:Dp|(?<![A-Za-z]))([CNW])(?![A-Za-z])')[0]
code_mufe = shades['shade_number'].where(shades['brand_name'] == 'MAKE UP FOR EVER').str.extract(r'^([RYN])\d')[0]
code_ysl = shades['shade_number'].where(shades['brand_name'] == 'Yves Saint Laurent').str.extract(r'^(B)\d')[0]
code_found = code_mufe.fillna(code_ysl).fillna(code_cnw).where(shades['brand_name'] != 'MAC')

has_phrase = shades['undertone_term'].notna()
has_word = word_found.notna()
has_code = code_found.notna()

shades['undertone_source'] = np.select([has_phrase, has_word, has_code], ['phrase', 'word', 'code'], default='none')
shades['undertone_brand_term'] = np.select([has_phrase, has_word, has_code], [shades['undertone_term'], word_found, code_found], default=None)

print(shades['undertone_source'].value_counts())

undertone_source
phrase    4836
none      1042
word       847
code        66
Name: count, dtype: int64


In [39]:
lookup = shades.drop_duplicates('shade_key')[['shade_key', 'undertone_brand_term', 'undertone_source']]
lookup.to_csv('../data/processed/undertone_lookup.csv', index=False)

with engine.begin() as conn:
    conn.execute(text('truncate table raw.undertone_lookup'))
lookup.to_sql('undertone_lookup', engine, schema='raw', if_exists='append', index=False)

print(len(lookup), 'rows in raw.undertone_lookup')

6779 rows in raw.undertone_lookup


## Undertone label mapping

In [42]:
terms = pd.read_sql(
    """
    select undertone_brand_term, count(*) as shades
    from dev.int_shades
    where undertone_brand_term is not null
    group by undertone_brand_term
    order by shades desc
    """,
    engine,
)

warm_words = {'warm', 'golden', 'gold', 'yellow', 'peach', 'peachy', 'orange', 'amber', 'bronze', 'caramel', 'ochre', 'w', 'y'}
cool_words = {'cool', 'pink', 'rosy', 'rosey', 'rose', 'red', 'reddish', 'blue', 'mauve', 'plum', 'lightpink', 'c', 'r'}
neutral_words = {'neutral', 'beige', 'n', 'b'}

def suggest_label(term):
    words = set(re.findall(r'[a-z]+', term.lower()))
    warm = words & warm_words
    cool = words & cool_words
    neutral = words & neutral_words
    olive = words & {'olive', 'green', 'goldenolive'}
    intensity = (words & {'subtle', 'slightly'}) and not (words & {'warm', 'cool'})
    if 'warm' in words and cool and 'cool' not in words:
        return 'warm'
    if warm and cool:
        return 'unlabelled'
    if not warm and not cool:
        return 'neutral' if (neutral or olive) else 'unlabelled'
    direction = 'warm' if warm else 'cool'
    if neutral or intensity:
        return direction + ' neutral'
    return direction

terms['undertone_label'] = terms['undertone_brand_term'].apply(suggest_label)
terms.to_csv('../02_dbt/seeds/undertone_label_map.csv', index=False)

print(len(terms), 'distinct terms,', (terms['undertone_label'] == 'check').sum(), 'marked check')
terms.head(40)

232 distinct terms, 0 marked check


,undertone_brand_term,shades,undertone_label
0,neutral,1340,neutral
1,warm,637,warm
2,golden,566,warm
3,cool,444,cool
4,yellow,303,warm
5,pink,289,cool
6,peach,209,warm
7,red,143,cool
8,cool neutral,112,cool neutral
9,warm neutral,109,warm neutral


In [43]:
pd.set_option('display.max_rows', None)
terms[terms['undertone_label'] == 'check']

,undertone_brand_term,shades,undertone_label


## Price tier 

In [44]:
brands = pd.read_sql(
    "select brand_name, count(*) as products from dev.int_products group by brand_name order by brand_name",
    engine,
)
brands['price_tier'] = ''
brands.to_csv('../02_dbt/seeds/brand_price_tiers.csv', index=False)
print(len(brands), 'brands')

103 brands
